# Projet : faire passer une courbe par des points

On connaît quelques points d'une courbe et on cherche le polynôme de degré 3 qui passe exactement par ces points. Le projet reprend les notions des cinq TP :

- **Matrices et systèmes linéaires** (TP4) : chaque point donne une équation, on résout le système par la méthode de Gauss.
- **Calcul numérique et flottants** (TP1) : on compare le résultat en nombres décimaux et le résultat exact.
- **Arithmétique** (TP2) : on utilise le PGCD pour vérifier que les fractions sont simplifiées.
- **Polynômes et racines** (TP3) : on évalue le polynôme, on le dérive et on cherche sa racine.
- **Géométrie du plan** (TP6) : on étudie le polygone formé par les points.

Exécutez les cellules dans l'ordre avec Maj+Entrée. Pour essayer d'autres points, changez la liste `points` dans la partie 1.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

## 1. Les points

On cherche $P(x) = c_0 + c_1 x + c_2 x^{2} + c_3 x^{3}$ avec $P(x_i) = y_i$ pour chaque point. Il faut autant de points que de coefficients : ici quatre.

In [2]:
points = [(0, -5), (1, -6), (2, -1), (3, 16)]
print(points)

[(0, -5), (1, -6), (2, -1), (3, 16)]


## 2. Le système linéaire (matrices)

Chaque point $(x, y)$ donne l'équation $c_0 + c_1 x + c_2 x^{2} + c_3 x^{3} = y$. L'ensemble forme un système dont la matrice contient les puissances de $x$. On le résout par le pivot de Gauss avec pivot partiel.

In [3]:
def resoudre(A, b):
    n = len(A)
    M = []
    for ligne in A:
        M.append(ligne[:])
    y = b[:]
    for k in range(n):
        p = k
        for i in range(k + 1, n):
            if abs(M[i][k]) > abs(M[p][k]):
                p = i
        M[k], M[p] = M[p], M[k]
        y[k], y[p] = y[p], y[k]
        if M[k][k] == 0:
            raise ValueError("système singulier")
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] = M[i][j] - facteur * M[k][j]
            y[i] = y[i] - facteur * y[k]
    x = [0] * n
    for i in range(n - 1, -1, -1):
        s = y[i]
        for j in range(i + 1, n):
            s = s - M[i][j] * x[j]
        x[i] = s / M[i][i]
    return x

def construire_systeme(points):
    n = len(points)
    A = []
    b = []
    for x, y in points:
        ligne = []
        for j in range(n):
            ligne.append(x ** j)
        A.append(ligne)
        b.append(y)
    return A, b

A, b = construire_systeme(points)
print(A)
print(b)

coefficients = resoudre(A, b)
print(coefficients)
print(np.linalg.solve(np.array(A, dtype=float), np.array(b, dtype=float)))

[[1, 0, 0, 0], [1, 1, 1, 1], [1, 2, 4, 8], [1, 3, 9, 27]]
[-5, -6, -1, 16]
[-5.0, -2.0, -0.0, 1.0]
[-5. -2. -0.  1.]


## 3. Les flottants et le calcul exact

Les nombres décimaux de l'ordinateur sont approchés : on ne doit pas les comparer avec `==`. On vérifie donc nos coefficients avec une tolérance, puis on refait le calcul en fractions, sans aucun arrondi.

In [4]:
print(0.1 + 0.2 == 0.3)

def presque_egaux(a, b, tol=1e-9):
    return abs(a - b) < tol

print(presque_egaux(0.1 + 0.2, 0.3))

attendus = [-5, -2, 0, 1]
for i in range(4):
    print(coefficients[i], presque_egaux(coefficients[i], attendus[i]))

def coefficients_exacts(points):
    points_exacts = []
    for x, y in points:
        points_exacts.append((Fraction(x), Fraction(y)))
    A, b = construire_systeme(points_exacts)
    return resoudre(A, b)

print(coefficients_exacts(points))

False
True
-5.0 True
-2.0 True
-0.0 True
1.0 True
[Fraction(-5, 1), Fraction(-2, 1), Fraction(0, 1), Fraction(1, 1)]


## 4. L'arithmétique

Avec d'autres points, les coefficients ne sont plus entiers. Le PGCD, calculé par l'algorithme d'Euclide, permet de vérifier que chaque fraction est bien simplifiée.

In [5]:
def pgcd(a, b):
    a = abs(a)
    b = abs(b)
    while b != 0:
        a, b = b, a % b
    return a

def simplifier(num, den):
    g = pgcd(num, den)
    return num // g, den // g

print(simplifier(6, 4))

points2 = [(0, 1), (1, 2), (2, 4)]
c2 = coefficients_exacts(points2)
print(c2)
for c in c2:
    print(c, pgcd(c.numerator, c.denominator))

(3, 2)
[Fraction(1, 1), Fraction(1, 2), Fraction(1, 2)]
1 1
1/2 1
1/2 1


## 5. Le polynôme et sa racine

On évalue le polynôme par la méthode de Horner, on le dérive, puis on cherche sa racine de deux façons : par la méthode de Newton et par dichotomie.

In [6]:
def evaluer(P, x):
    resultat = 0
    for coef in reversed(P):
        resultat = resultat * x + coef
    return resultat

def deriver(P):
    resultat = []
    for i in range(1, len(P)):
        resultat.append(i * P[i])
    return resultat

def newton(P, depart, tolerance):
    dP = deriver(P)
    x = depart
    tours = 0
    while tours < 100:
        nouveau = x - evaluer(P, x) / evaluer(dP, x)
        tours = tours + 1
        if abs(nouveau - x) < tolerance:
            return nouveau, tours
        x = nouveau
    return x, tours

def dichotomie(f, a, b, tolerance):
    while b - a > tolerance:
        m = (a + b) / 2
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
    return (a + b) / 2

def f(x):
    return evaluer(coefficients, x)

for x, y in points:
    print(x, y, evaluer(coefficients, x))

print(deriver(coefficients))

racine, tours = newton(coefficients, 2, 1e-10)
print(racine, tours)
print(dichotomie(f, 2, 3, 1e-10))
print(f(racine))

0 -5 -5.0
1 -6 -6.0
2 -1 -1.0
3 16 16.0
[-2.0, -0.0, 3.0]
2.0945514815423265 5
2.0945514815684874
-1.7763568394002505e-15


## 6. La géométrie

Les quatre points forment un polygone. On calcule son aire avec la formule du lacet, on regarde dans quel sens tourne le trajet, on mesure la distance d'un point à une droite et on teste si le point de la racine est à l'intérieur.

In [7]:
def vecteur(A, B):
    return (B[0] - A[0], B[1] - A[1])

def norme(u):
    return math.sqrt(u[0] * u[0] + u[1] * u[1])

def determinant(u, v):
    return u[0] * v[1] - u[1] * v[0]

def orientation(A, B, C):
    d = determinant(vecteur(A, B), vecteur(A, C))
    if d > 0:
        return 1
    if d < 0:
        return -1
    return 0

def aire(polygone):
    n = len(polygone)
    total = 0
    for i in range(n):
        x1, y1 = polygone[i]
        x2, y2 = polygone[(i + 1) % n]
        total = total + x1 * y2 - x2 * y1
    return abs(total) / 2

def distance_point_droite(P, A, B):
    AB = vecteur(A, B)
    AP = vecteur(A, P)
    return abs(determinant(AB, AP)) / norme(AB)

def dedans(P, polygone):
    n = len(polygone)
    croisements = 0
    for i in range(n):
        S = polygone[i]
        T = polygone[(i + 1) % n]
        if (S[1] <= P[1] < T[1]) or (T[1] <= P[1] < S[1]):
            x = S[0] + (P[1] - S[1]) * (T[0] - S[0]) / (T[1] - S[1])
            if x > P[0]:
                croisements = croisements + 1
    return croisements % 2 == 1

print(aire(points))

for i in range(len(points) - 2):
    print(points[i], points[i + 1], points[i + 2], orientation(points[i], points[i + 1], points[i + 2]))

print(orientation((0, 0), (1, 1), (2, 2)))
print(distance_point_droite(points[2], points[1], points[3]))
print(dedans((racine, 0), points))

18.0
(0, -5) (1, -6) (2, -1) 1
(1, -6) (2, -1) (3, 16) 1
0
0.5432144762551112
False


## 7. Le graphique

La courbe passe par les quatre points (en rouge) et coupe l'axe horizontal à la racine (en vert). Le polygone des points est en pointillés.

In [8]:
xs = []
ys = []
for i in range(301):
    x = -0.5 + 4 * i / 300
    xs.append(x)
    ys.append(evaluer(coefficients, x))
plt.plot(xs, ys, "b-", label="polynôme")

px = []
py = []
for x, y in points + [points[0]]:
    px.append(x)
    py.append(y)
plt.plot(px, py, "k--", label="polygone")

for x, y in points:
    plt.plot(x, y, "ro")
plt.plot(racine, 0, "go", label="racine")

plt.axhline(0, color="black")
plt.legend()
plt.show()

## Conclusion

En résolvant un système linéaire, on a retrouvé le polynôme $x^{3} - 2x - 5$ qui passe par les quatre points, d'abord avec des nombres décimaux puis exactement avec des fractions. On a ensuite cherché sa racine, environ $2{,}0945515$, et étudié la géométrie du polygone formé par les points.